# Load SAP Planning Demo

Attach `SAPPlanningLakehouse` as the default lakehouse before running. This notebook creates SAP-style dummy actuals, baseline plan, revised plan, and gold planning tables.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.sql('CREATE DATABASE IF NOT EXISTS bronze')
spark.sql('CREATE DATABASE IF NOT EXISTS silver')
spark.sql('CREATE DATABASE IF NOT EXISTS gold')


In [ ]:
materials = [
    ('MAT-100','Contoso Smart Sensor','Electronics',42.0,89.0,'A',365),
    ('MAT-200','Fabrikam Control Valve','Industrial',76.0,141.0,'A',540),
    ('MAT-300','Northwind Pump Kit','Industrial',118.0,225.0,'B',720),
    ('MAT-400','AdventureWorks Battery Pack','Electronics',35.0,79.0,'A',270),
    ('MAT-500','Graphic Design Packaging','Packaging',4.0,9.0,'C',180),
    ('MAT-600','Tailspin Safety Harness','Safety',28.0,62.0,'B',1095)
]
plants = [('PL-10','Seattle Distribution Center','West','Alex Planner'),('PL-20','Chicago Manufacturing Hub','Central','Casey Planner'),('PL-30','Atlanta Fulfillment Center','East','Jordan Planner')]
suppliers = [('SUP-01','Alpine Components','US','Low',12),('SUP-02','Blue Yonder Metals','MX','Medium',18),('SUP-03','Contoso Electronics','TW','High',31),('SUP-04','Datum Packaging','US','Low',7)]

inventory = [
('2026-09-01','MAT-100','PL-10',820,340,20),('2026-09-01','MAT-100','PL-20',510,300,0),('2026-09-01','MAT-100','PL-30',390,310,10),
('2026-09-01','MAT-200','PL-10',240,150,5),('2026-09-01','MAT-200','PL-20',740,290,20),('2026-09-01','MAT-200','PL-30',180,130,0),
('2026-09-01','MAT-300','PL-10',160,120,0),('2026-09-01','MAT-300','PL-20',210,180,5),('2026-09-01','MAT-300','PL-30',130,100,0),
('2026-09-01','MAT-400','PL-10',960,400,30),('2026-09-01','MAT-400','PL-20',460,390,0),('2026-09-01','MAT-400','PL-30',520,310,0),
('2026-09-01','MAT-500','PL-10',5100,1800,0),('2026-09-01','MAT-500','PL-20',2700,1300,0),('2026-09-01','MAT-500','PL-30',3100,1500,0),
('2026-09-01','MAT-600','PL-10',430,260,0),('2026-09-01','MAT-600','PL-20',220,190,0),('2026-09-01','MAT-600','PL-30',290,210,0)
]

sales_orders = [('SO-10001','2026-09-02','2026-09-12','MAT-100','PL-10','Enterprise',280),('SO-10002','2026-09-03','2026-09-14','MAT-100','PL-30','Commercial',220),('SO-10003','2026-09-03','2026-09-20','MAT-200','PL-30','Enterprise',160),('SO-10004','2026-09-04','2026-09-16','MAT-300','PL-20','Enterprise',190),('SO-10005','2026-09-05','2026-09-18','MAT-400','PL-20','Consumer',360),('SO-10006','2026-09-05','2026-09-18','MAT-600','PL-20','Enterprise',180),('SO-10007','2026-09-07','2026-09-22','MAT-500','PL-10','Commercial',900),('SO-10008','2026-09-07','2026-09-24','MAT-300','PL-30','Enterprise',115)]
purchase_orders = [('PO-9001','2026-08-22','2026-09-15','MAT-100','PL-30','SUP-03',300,42.0),('PO-9002','2026-08-24','2026-09-17','MAT-200','PL-30','SUP-02',260,76.0),('PO-9003','2026-08-26','2026-09-25','MAT-300','PL-20','SUP-02',240,118.0),('PO-9004','2026-08-26','2026-09-13','MAT-400','PL-20','SUP-03',420,35.0),('PO-9005','2026-08-28','2026-09-08','MAT-500','PL-10','SUP-04',1100,4.0),('PO-9006','2026-08-29','2026-09-28','MAT-600','PL-20','SUP-01',260,28.0)]
goods_receipts = [('GR-7001','PO-8801','2026-08-20','MAT-100','PL-10','SUP-03',400,5),('GR-7002','PO-8802','2026-08-24','MAT-200','PL-20','SUP-02',300,2),('GR-7003','PO-8803','2026-08-26','MAT-400','PL-10','SUP-03',500,7),('GR-7004','PO-8804','2026-08-27','MAT-500','PL-30','SUP-04',1500,0),('GR-7005','PO-8805','2026-08-30','MAT-600','PL-10','SUP-01',300,1)]
forecast = [('2026-10','MAT-100','PL-10',710,760),('2026-10','MAT-100','PL-20',490,560),('2026-10','MAT-100','PL-30',560,680),('2026-10','MAT-200','PL-10',320,360),('2026-10','MAT-200','PL-20',620,660),('2026-10','MAT-200','PL-30',310,420),('2026-10','MAT-300','PL-10',230,250),('2026-10','MAT-300','PL-20',390,470),('2026-10','MAT-300','PL-30',240,330),('2026-10','MAT-400','PL-10',790,840),('2026-10','MAT-400','PL-20',610,790),('2026-10','MAT-400','PL-30',470,520),('2026-10','MAT-500','PL-10',2700,2900),('2026-10','MAT-500','PL-20',2100,2200),('2026-10','MAT-500','PL-30',2300,2400),('2026-10','MAT-600','PL-10',420,460),('2026-10','MAT-600','PL-20',360,470),('2026-10','MAT-600','PL-30',330,370)]
scenarios = [('BASELINE','Current baseline plan','Baseline'),('REV1','Planner revised replenishment','Revised')]
assumptions = [('BASELINE','MAT-100','PL-30',0.00,10,31,450),('BASELINE','MAT-200','PL-30',0.00,9,18,300),('BASELINE','MAT-300','PL-20',0.00,8,18,260),('BASELINE','MAT-400','PL-20',0.00,10,31,520),('BASELINE','MAT-600','PL-20',0.00,7,12,260),('REV1','MAT-100','PL-30',0.08,15,36,620),('REV1','MAT-200','PL-30',0.05,12,22,410),('REV1','MAT-300','PL-20',0.10,12,24,390),('REV1','MAT-400','PL-20',0.12,16,38,760),('REV1','MAT-600','PL-20',0.04,10,14,340)]


In [ ]:
def write_table(data, schema, table_name):
    df = spark.createDataFrame(data, schema=schema)
    df.write.mode('overwrite').format('delta').saveAsTable(table_name)
    return df

write_table(materials, 'material_id string, material_name string, product_family string, unit_cost double, unit_price double, abc_class string, shelf_life_days int', 'bronze.materials')
write_table(plants, 'plant_id string, plant_name string, region string, planning_owner string', 'bronze.plants')
write_table(suppliers, 'supplier_id string, supplier_name string, country string, risk_tier string, standard_lead_time_days int', 'bronze.suppliers')
write_table(inventory, 'snapshot_date string, material_id string, plant_id string, on_hand_qty int, allocated_qty int, quality_hold_qty int', 'bronze.inventory_snapshots')
write_table(sales_orders, 'order_id string, order_date string, requested_ship_date string, material_id string, plant_id string, customer_segment string, open_qty int', 'bronze.sales_orders')
write_table(purchase_orders, 'po_id string, po_date string, expected_receipt_date string, material_id string, plant_id string, supplier_id string, open_qty int, unit_cost double', 'bronze.purchase_orders')
write_table(goods_receipts, 'receipt_id string, po_id string, receipt_date string, material_id string, plant_id string, supplier_id string, received_qty int, days_late int', 'bronze.goods_receipts')
write_table(forecast, 'forecast_month string, material_id string, plant_id string, baseline_demand_qty int, consensus_demand_qty int', 'bronze.demand_forecast')
write_table(scenarios, 'scenario_id string, scenario_name string, scenario_type string', 'bronze.planning_scenarios')
write_table(assumptions, 'scenario_id string, material_id string, plant_id string, demand_uplift_pct double, safety_stock_days int, lead_time_override_days int, reorder_qty int', 'bronze.planning_assumptions')


In [ ]:
spark.sql('''
CREATE OR REPLACE TABLE silver.inventory_position AS
SELECT
  i.snapshot_date,
  i.material_id,
  m.material_name,
  m.product_family,
  m.unit_cost,
  i.plant_id,
  p.plant_name,
  p.region,
  p.planning_owner,
  i.on_hand_qty,
  i.allocated_qty,
  i.quality_hold_qty,
  i.on_hand_qty - i.allocated_qty - i.quality_hold_qty AS available_qty
FROM bronze.inventory_snapshots i
JOIN bronze.materials m ON i.material_id = m.material_id
JOIN bronze.plants p ON i.plant_id = p.plant_id
''')

spark.sql('''
CREATE OR REPLACE TABLE silver.open_demand AS
SELECT material_id, plant_id, SUM(open_qty) AS open_sales_order_qty
FROM bronze.sales_orders
GROUP BY material_id, plant_id
''')

spark.sql('''
CREATE OR REPLACE TABLE silver.open_supply AS
SELECT material_id, plant_id, supplier_id, SUM(open_qty) AS open_purchase_order_qty
FROM bronze.purchase_orders
GROUP BY material_id, plant_id, supplier_id
''')

spark.sql('''
CREATE OR REPLACE TABLE silver.supplier_performance AS
SELECT
  s.supplier_id,
  s.supplier_name,
  s.country,
  s.risk_tier,
  s.standard_lead_time_days,
  COALESCE(AVG(g.days_late), 0) AS avg_days_late,
  CASE WHEN s.risk_tier = 'High' OR COALESCE(AVG(g.days_late), 0) >= 5 THEN 'High'
       WHEN s.risk_tier = 'Medium' OR COALESCE(AVG(g.days_late), 0) >= 2 THEN 'Medium'
       ELSE 'Low' END AS supplier_delay_risk
FROM bronze.suppliers s
LEFT JOIN bronze.goods_receipts g ON s.supplier_id = g.supplier_id
GROUP BY s.supplier_id, s.supplier_name, s.country, s.risk_tier, s.standard_lead_time_days
''')


In [ ]:
spark.sql('''
CREATE OR REPLACE TABLE gold.gold_inventory_position AS
SELECT
  ip.*,
  f.forecast_month,
  to_date(concat(f.forecast_month, '-01')) AS forecast_month_start,
  CAST(date_format(to_date(concat(f.forecast_month, '-01')), 'yyyyMM') AS INT) AS time_period_id,
  COALESCE(d.open_sales_order_qty, 0) AS open_sales_order_qty,
  COALESCE(f.baseline_demand_qty, 0) AS baseline_demand_qty,
  COALESCE(f.consensus_demand_qty, 0) AS consensus_demand_qty,
  COALESCE(s.open_purchase_order_qty, 0) AS open_purchase_order_qty,
  COALESCE(sp.supplier_delay_risk, 'Unknown') AS supplier_delay_risk,
  CASE WHEN ip.available_qty + COALESCE(s.open_purchase_order_qty, 0) < COALESCE(f.consensus_demand_qty, 0) THEN 'High'
       WHEN ip.available_qty < COALESCE(f.consensus_demand_qty, 0) * 0.6 THEN 'Medium'
       ELSE 'Low' END AS stockout_risk_before_plan
FROM silver.inventory_position ip
LEFT JOIN silver.open_demand d ON ip.material_id = d.material_id AND ip.plant_id = d.plant_id
LEFT JOIN bronze.demand_forecast f ON ip.material_id = f.material_id AND ip.plant_id = f.plant_id
LEFT JOIN silver.open_supply s ON ip.material_id = s.material_id AND ip.plant_id = s.plant_id
LEFT JOIN silver.supplier_performance sp ON s.supplier_id = sp.supplier_id
''')

spark.sql('CREATE OR REPLACE TABLE gold.gold_supplier_performance AS SELECT * FROM silver.supplier_performance')

spark.sql('''
CREATE OR REPLACE TABLE gold.gold_time_period AS
SELECT DISTINCT
  time_period_id,
  forecast_month_start AS period_start_date,
  YEAR(forecast_month_start) AS year,
  QUARTER(forecast_month_start) AS quarter,
  MONTH(forecast_month_start) AS month_number,
  date_format(forecast_month_start, 'MMMM') AS month_name,
  date_format(forecast_month_start, 'yyyy-MM') AS year_month
FROM gold.gold_inventory_position
WHERE forecast_month_start IS NOT NULL
''')

spark.sql('''
CREATE OR REPLACE TABLE gold.gold_replenishment_plan AS
SELECT
  a.scenario_id,
  sc.scenario_name,
  sc.scenario_type,
  a.material_id,
  m.material_name,
  m.product_family,
  a.plant_id,
  p.plant_name,
  p.region,
  p.planning_owner,
  a.demand_uplift_pct,
  a.safety_stock_days,
  a.lead_time_override_days,
  a.reorder_qty,
  a.reorder_qty * m.unit_cost AS working_capital_impact
FROM bronze.planning_assumptions a
JOIN bronze.planning_scenarios sc ON a.scenario_id = sc.scenario_id
JOIN bronze.materials m ON a.material_id = m.material_id
JOIN bronze.plants p ON a.plant_id = p.plant_id
''')

spark.sql('''
CREATE OR REPLACE TABLE gold.gold_plan_vs_actual AS
SELECT
  r.scenario_id,
  r.scenario_name,
  r.scenario_type,
  i.material_id,
  i.material_name,
  i.product_family,
  i.plant_id,
  i.plant_name,
  i.region,
  i.planning_owner,
  i.forecast_month,
  i.forecast_month_start,
  i.time_period_id,
  i.available_qty,
  i.open_purchase_order_qty,
  i.open_sales_order_qty,
  i.baseline_demand_qty,
  i.consensus_demand_qty,
  CAST(i.consensus_demand_qty * (1 + r.demand_uplift_pct) AS INT) AS planned_demand_qty,
  r.safety_stock_days,
  r.lead_time_override_days,
  r.reorder_qty,
  i.available_qty + i.open_purchase_order_qty + r.reorder_qty - CAST(i.consensus_demand_qty * (1 + r.demand_uplift_pct) AS INT) AS projected_inventory_after_plan,
  i.stockout_risk_before_plan,
  CASE WHEN i.available_qty + i.open_purchase_order_qty + r.reorder_qty < CAST(i.consensus_demand_qty * (1 + r.demand_uplift_pct) AS INT) THEN 'High'
       WHEN i.available_qty + i.open_purchase_order_qty + r.reorder_qty < CAST(i.consensus_demand_qty * (1 + r.demand_uplift_pct) AS INT) * 1.15 THEN 'Medium'
       ELSE 'Low' END AS stockout_risk_after_plan,
  r.working_capital_impact,
  i.supplier_delay_risk
FROM gold.gold_inventory_position i
JOIN gold.gold_replenishment_plan r ON i.material_id = r.material_id AND i.plant_id = r.plant_id
''')


In [ ]:
display(spark.sql('SELECT * FROM gold.gold_plan_vs_actual ORDER BY stockout_risk_after_plan DESC, working_capital_impact DESC'))
